# Notebook 2: GEDI UOI Signal Generation (GEE)

This notebook computes the raw Understory Openness Index (UOI) from GEDI L2B data.
It follows a strict modular structure:
1. Setup & Configuration
2. Methodological Logic (Functions)
3. Unit Tests
4. Execution (Asset Export)

In [ ]:
# =============================================================================
# BLOCK 1: SETUP AND CONFIGURATION
# =============================================================================
import ee

try:
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")

# Output destination
ASSET_ROOT = 'projects/quantum-bonus-434714-t2/assets/DefaunationFromSpace'

# Study Regions
CONGO_BBOX = ee.Geometry.Rectangle([8, -12, 35, 8])
AMAZON_BBOX = ee.Geometry.Rectangle([-73, -18, -44, 8])
BASINS = [('Congo', CONGO_BBOX), ('Amazon', AMAZON_BBOX)]

# Export resolution: 1km captures all GEDI spatial information.
EXPORT_SCALE = 1000

# Datasets
GEDI_L2B = 'LARSE/GEDI/GEDI02_B_002_MONTHLY'

# Date range
START_DATE = '2020-01-01'
END_DATE = '2023-12-31'

print("\u2713 Configuration loaded.")

In [ ]:
# =============================================================================
# BLOCK 2: METHODOLOGICAL LOGIC (FUNCTIONS)
# =============================================================================

def compute_native_uoi():
    """Calculates UOI and observation counts from raw GEDI L2B.
    
    No land cover or topographic masks are applied here. Raw GEDI
    footprints are aggregated to preserve maximum data fidelity.
    Forest and topo filtering are applied downstream in NB3 when
    joining with FRIP under a harmonized masking scheme.
    """
    gedi = ee.ImageCollection(GEDI_L2B).filterDate(START_DATE, END_DATE)
    
    def calc_uoi(img):
        pai = img.select('pai')
        pavd_z0 = img.select('pavd_z0')
        uoi = ee.Image(1).subtract(pavd_z0.divide(pai))
        return uoi.clamp(0, 1).rename('UOI')
    
    uoi_col = gedi.map(calc_uoi)
    
    mean_uoi = uoi_col.mean().rename('UOI_mean')
    count_n = uoi_col.count().rename('N')
    
    native_proj = gedi.first().projection()
    native_stack = ee.Image.cat([mean_uoi, count_n]).setDefaultProjection(native_proj)
    
    return native_stack, native_proj

def build_gedi_asset():
    """Builds the GEDI UOI asset at 1km export resolution.
    
    Aggregates 25m GEDI data to 1km using reduceResolution.
    At 25m -> 1km, each output pixel contains ~1,600 input pixels,
    safely under the 65,535 maxPixels limit.
    """
    native_stack, native_proj = compute_native_uoi()
    
    target_proj = native_proj.atScale(EXPORT_SCALE)
    
    agg_uoi = native_stack.select('UOI_mean').reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).setDefaultProjection(target_proj)
    
    agg_n = native_stack.select('N').reduceResolution(
        reducer=ee.Reducer.sum(),
        maxPixels=65535
    ).setDefaultProjection(target_proj)
    
    return ee.Image.cat([agg_uoi, agg_n])

print("\u2713 Methodological functions loaded.")

In [ ]:
# =============================================================================
# BLOCK 3: UNIT TESTS
# =============================================================================

def run_unit_tests():
    print("Running Unit Tests for GEDI UOI generation...")
    passed = 0
    
    try:
        print("  [1/2] Testing native UOI computation from GEDI L2B...")
        native_stack, native_proj = compute_native_uoi()
        native_bands = native_stack.bandNames().getInfo()
        assert 'UOI_mean' in native_bands, f"Missing UOI_mean. Got: {native_bands}"
        assert 'N' in native_bands, f"Missing N. Got: {native_bands}"
        passed += 1
        print(f"    \u2713 Native stack bands: {native_bands}")
        
        print("  [2/2] Testing 1km aggregated asset construction...")
        gedi_asset = build_gedi_asset()
        bands = gedi_asset.bandNames().getInfo()
        assert len(bands) == 2, f"Expected 2 bands, got {len(bands)}"
        assert 'UOI_mean' in bands, "Missing UOI_mean"
        assert 'N' in bands, "Missing N"
        passed += 1
        print(f"    \u2713 Aggregated asset bands: {bands}")
        
        print(f"\n{'='*60}")
        print(f"  \u2713 ALL {passed} TESTS PASSED")
        print(f"  No masks applied — raw GEDI UOI for maximum fidelity.")
        print(f"  Forest/topo filtering deferred to NB3.")
        print(f"{'='*60}")
        
    except Exception as e:
        print(f"  \u2717 Error ({passed} passed): {e}")

run_unit_tests()

In [ ]:
# =============================================================================
# BLOCK 4: EXECUTION (ASSET EXPORT)
# =============================================================================

BASINS = [('Congo', CONGO_BBOX), ('Amazon', AMAZON_BBOX)]

def safe_start(task, asset_id):
    """Deletes existing asset if present, then starts the export task."""
    try:
        ee.data.deleteAsset(asset_id)
        print(f"    Deleted existing: {asset_id.split('/')[-1]}")
    except Exception:
        pass
    task.start()

def export_gedi(dry_run=True):
    print("Configuring GEDI UOI exports (1km, per basin)...")
    
    gedi_asset = build_gedi_asset()
    
    tasks = []
    for basin_name, basin_geom in BASINS:
        asset_id = f'{ASSET_ROOT}/Openness_raw/GEDI_1km_{basin_name}'
        task = ee.batch.Export.image.toAsset(
            image=gedi_asset,
            description=f'GEDI_1km_{basin_name}',
            assetId=asset_id,
            region=basin_geom,
            scale=EXPORT_SCALE,
            crs='EPSG:4326',
            maxPixels=1e13
        )
        tasks.append((task, asset_id))
    
    print(f"\u2713 Configured {len(tasks)} export tasks.")
    if dry_run:
        print("DRY RUN: Tasks created but not started. Call export_gedi(dry_run=False) to begin.")
    else:
        for task, asset_id in tasks:
            safe_start(task, asset_id)
        print(f"\u2713 {len(tasks)} tasks started! Monitor at https://code.earthengine.google.com/tasks")

# To execute the export, set dry_run=False
export_gedi(dry_run=True)